# Modulo 4 · Lezione 2 — Query di base con SELECT

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Istruzione / clausola | A cosa serve |
|---|---------------------|----------------|
| 1 | `SELECT` di base | Scegliere le colonne |
| 2 | `WHERE` | Filtrare le righe |
| 3 | `ORDER BY` | Ordinare i risultati |
| 4 | `LIMIT` e `OFFSET` | Paginare i risultati |
| 5 | `DISTINCT` | Eliminare i duplicati |
| 6 | Funzioni di aggregazione | `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` |
| 7 | Alias con `AS` | Nomi leggibili nel risultato |
| 8 | Pratica | Esplorare il database del negozio |

> 💡 Useremo il database `tienda.db` creato nella Lezione 1. Eseguila prima se non ce l'hai ancora.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")
conn.row_factory = sqlite3.Row   # accesso alle colonne per nome

def q(sql, *params):
    """Esegue una query e restituisce un DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

# Verificare che le tabelle esistano
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("Tabelle disponibili:", tablas['name'].tolist())

---
## 1. `SELECT` di base

```sql
-- Tutte le colonne
SELECT * FROM tabla;

-- Colonne specifiche
SELECT col1, col2 FROM tabla;

-- Espressioni calcolate
SELECT nombre, precio * 1.21 AS precio_con_iva FROM productos;
```

La struttura di base di una `SELECT` completa segue questo **ordine fisso**:

```sql
SELECT   colonne             -- cosa voglio vedere?
FROM     tabella             -- da dove?
WHERE    condizione          -- quali righe?
ORDER BY colonna [ASC|DESC]  -- in che ordine?
LIMIT    n OFFSET m;         -- quante?
```

In [ ]:
# Tutte le colonne
print("=== SELECT * ===")
display(q("SELECT * FROM clientes"))

# Colonne specifiche
print("\n=== Colonne specifiche ===")
display(q("SELECT nombre, email FROM clientes"))

# Espressione calcolata: prezzo con IVA
print("\n=== Prezzo con IVA (21%) ===")
display(q("""
    SELECT nombre,
           precio                        AS precio_sin_iva,
           ROUND(precio * 1.21, 2)       AS precio_con_iva
    FROM productos
    LIMIT 5
"""))

---
## 2. `WHERE` — Filtrare le righe

```sql
WHERE condizione
```

**Operatori di confronto:**

| Operatore | Significato | Esempio |
|----------|-------------|--------|
| `=` | Uguale | `precio = 99.99` |
| `!=` / `<>` | Diverso | `estado != 'cancelado'` |
| `>`, `<`, `>=`, `<=` | Confronto numerico/testuale | `precio > 100` |
| `BETWEEN a AND b` | Intervallo incluso | `precio BETWEEN 50 AND 200` |
| `IN (v1, v2, ...)` | In un elenco | `ciudad IN ('Madrid', 'Sevilla')` |
| `NOT IN (...)` | Fuori dall'elenco | `estado NOT IN ('cancelado')` |
| `LIKE 'pattern'` | Corrispondenza testuale | `nombre LIKE 'A%'` |
| `IS NULL` | Valore nullo | `descripcion IS NULL` |
| `IS NOT NULL` | Non nullo | `email IS NOT NULL` |

**Pattern `LIKE`:**
- `%` → qualsiasi sequenza di caratteri (anche nessuno)
- `_` → esattamente un carattere

**Operatori logici:** `AND`, `OR`, `NOT`

In [ ]:
# Confronto semplice
print("=== Prodotti con prezzo < 50 ===")
display(q("SELECT nombre, precio FROM productos WHERE precio < 50 ORDER BY precio"))

# BETWEEN
print("\n=== Prezzo tra 50 e 200 ===")
display(q("SELECT nombre, precio FROM productos WHERE precio BETWEEN 50 AND 200"))

# IN
print("\n=== Ordini in attesa o spediti ===")
display(q("SELECT id, cliente_id, estado FROM pedidos WHERE estado IN ('pendiente','enviado')"))

# LIKE
print("\n=== Prodotti il cui nome contiene 'Pro' ===")
display(q("SELECT nombre, precio FROM productos WHERE nombre LIKE '%Pro%'"))

# AND / OR
print("\n=== Elettronica con stock > 100 ===")
display(q("""
    SELECT nombre, precio, stock
    FROM productos
    WHERE categoria_id = 1
      AND stock > 100
"""))

---
## 3. `ORDER BY` — Ordinare i risultati

```sql
ORDER BY colonna1 [ASC|DESC], colonna2 [ASC|DESC]
```

- `ASC` (predefinito): minore → maggiore, A → Z
- `DESC`: maggiore → minore, Z → A
- Puoi ordinare per più colonne: prima per la prima, e in caso di parità per la seconda.

In [ ]:
# I più costosi per primi
print("=== Top 5 prodotti più costosi ===")
display(q("""
    SELECT nombre, precio, stock
    FROM productos
    ORDER BY precio DESC
    LIMIT 5
"""))

# Ordinare per categoria e all'interno di ciascuna per prezzo
print("\n=== Per categoria e prezzo ===")
display(q("""
    SELECT categoria_id, nombre, precio
    FROM productos
    ORDER BY categoria_id ASC, precio DESC
"""))

# Ordinare i clienti per città e nome
print("\n=== Clienti ordinati ===")
display(q("SELECT nombre, ciudad FROM clientes ORDER BY ciudad, nombre"))

---
## 4. `LIMIT` e `OFFSET` — Paginare i risultati

```sql
LIMIT n           -- restituisce solo i primi n risultati
LIMIT n OFFSET m  -- salta i primi m e restituisce i successivi n
```

Molto utile per implementare la paginazione nelle applicazioni:

```
Pagina 1: LIMIT 10 OFFSET 0
Pagina 2: LIMIT 10 OFFSET 10
Pagina 3: LIMIT 10 OFFSET 20
```

In [ ]:
PAGINA_SIZE = 3

for pagina in range(1, 5):
    offset = (pagina - 1) * PAGINA_SIZE
    resultado = q("""
        SELECT id, nombre, precio
        FROM productos
        ORDER BY precio DESC
        LIMIT ? OFFSET ?
    """, PAGINA_SIZE, offset)
    if resultado.empty:
        break
    print(f"Pagina {pagina}:")
    display(resultado)

---
## 5. `DISTINCT` — Eliminare i duplicati

```sql
SELECT DISTINCT columna FROM tabla;
SELECT DISTINCT col1, col2 FROM tabla;  -- combinazioni uniche
```

In [ ]:
# Senza DISTINCT: possono esserci città ripetute
print("=== Città (con duplicati) ===")
display(q("SELECT ciudad FROM clientes ORDER BY ciudad"))

# Con DISTINCT: ogni valore una sola volta
print("\n=== Città uniche ===")
display(q("SELECT DISTINCT ciudad FROM clientes ORDER BY ciudad"))

# DISTINCT con più colonne: combinazioni uniche
print("\n=== Combinazioni uniche stato–cliente ===")
display(q("SELECT DISTINCT cliente_id, estado FROM pedidos ORDER BY cliente_id"))

---
## 6. Funzioni di aggregazione

Le funzioni di aggregazione calcolano un unico valore a partire da un insieme di righe.

| Funzione | Restituisce | NULL |
|---------|----------|------|
| `COUNT(*)` | Numero di righe | Conta tutto |
| `COUNT(col)` | Righe dove col non è NULL | Ignora i NULL |
| `SUM(col)` | Somma dei valori | Ignora i NULL |
| `AVG(col)` | Media aritmetica | Ignora i NULL |
| `MIN(col)` | Valore minimo | Ignora i NULL |
| `MAX(col)` | Valore massimo | Ignora i NULL |

```sql
-- Senza GROUP BY: calcola su TUTTA la tabella
SELECT COUNT(*), AVG(precio), MAX(precio) FROM productos;

-- Con GROUP BY (Lezione 3): calcola per gruppo
SELECT categoria_id, AVG(precio) FROM productos GROUP BY categoria_id;
```

In [ ]:
# Statistiche generali dei prodotti
print("=== Statistiche dei prodotti ===")
display(q("""
    SELECT
        COUNT(*)                AS total_productos,
        ROUND(AVG(precio), 2)   AS precio_medio,
        MIN(precio)             AS precio_minimo,
        MAX(precio)             AS precio_maximo,
        SUM(stock)              AS stock_total
    FROM productos
"""))

# COUNT differenza * vs colonna
print("\n=== COUNT(*) vs COUNT(colonna) ===")
display(q("""
    SELECT
        COUNT(*)           AS total_filas,
        COUNT(descripcion) AS con_descripcion
    FROM categorias
"""))

# Valore totale dell'inventario
print("\n=== Valore totale dell'inventario ===")
display(q("""
    SELECT
        COUNT(*)                          AS num_productos,
        ROUND(SUM(precio * stock), 2)     AS valor_inventario
    FROM productos
    WHERE stock > 0
"""))

# Prezzo medio dei prodotti con prezzo > 100
print("\n=== Prezzo medio dei prodotti premium (>100€) ===")
display(q("""
    SELECT ROUND(AVG(precio), 2) AS precio_medio_premium
    FROM productos
    WHERE precio > 100
"""))

---
## 7. Alias con `AS`

Gli alias assegnano un nome temporaneo a una colonna o espressione nel risultato.

```sql
SELECT nombre AS producto,
       precio AS coste,
       precio * stock AS valor_stock
FROM productos;

-- In SQLite, AS è facoltativo (ma è buona pratica includerlo)
SELECT nombre producto FROM productos;
```

> Gli alias **non possono** essere usati in `WHERE` (il motore valuta `WHERE` prima di calcolare le colonne). Possono invece essere usati in `ORDER BY`.

In [ ]:
# Alias in colonne calcolate
display(q("""
    SELECT
        nombre                          AS producto,
        precio                          AS precio_base,
        ROUND(precio * 1.21, 2)         AS precio_iva,
        stock                           AS unidades,
        ROUND(precio * stock, 2)        AS valor_en_almacen
    FROM productos
    ORDER BY valor_en_almacen DESC
    LIMIT 8
"""))

---
## 8. Pratica — Esplorare il database

---

### Esercizio 1 — Filtri di base

**Consegna:** Scrivi una query per ogni caso:

1. Clienti di Madrid.
2. Prodotti con stock tra 50 e 150 unità.
3. Ordini del mese di gennaio 2025 (suggerimento: `LIKE '2025-01%'`).
4. Prodotti il cui nome inizia per vocale (A, E, I, O, U).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
# 1. Clienti di Madrid
display(q("SELECT nombre, email FROM clientes WHERE ciudad = 'Madrid'"))

# 2. Stock tra 50 e 150
display(q("SELECT nombre, stock FROM productos WHERE stock BETWEEN 50 AND 150 ORDER BY stock"))

# 3. Ordini di gennaio 2025
display(q("SELECT id, cliente_id, fecha, estado FROM pedidos WHERE fecha LIKE '2025-01%'"))

# 4. Nome che inizia per vocale
display(q("""
    SELECT nombre FROM productos
    WHERE nombre LIKE 'A%'
       OR nombre LIKE 'E%'
       OR nombre LIKE 'I%'
       OR nombre LIKE 'O%'
       OR nombre LIKE 'U%'
    ORDER BY nombre
"""))
```
</details>

---
### Esercizio 2 — Ordinare e paginare

**Consegna:**

1. Mostra i tre prodotti con maggior valore di inventario (`precio × stock`).
2. Elenca i clienti ordinati per città (A→Z) e all'interno di ciascuna città per nome (A→Z).
3. Implementa una funzione `paginar(tabla, columna_orden, pagina, por_pagina=3)` che restituisca un DataFrame con la pagina corrispondente.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
# 1. Top 3 per valore di inventario
display(q("""
    SELECT nombre,
           precio,
           stock,
           ROUND(precio * stock, 2) AS valor_inventario
    FROM productos
    ORDER BY valor_inventario DESC
    LIMIT 3
"""))

# 2. Clienti ordinati
display(q("SELECT nombre, ciudad FROM clientes ORDER BY ciudad ASC, nombre ASC"))

# 3. Funzione di paginazione
def paginar(tabla, columna_orden, pagina, por_pagina=3):
    offset = (pagina - 1) * por_pagina
    return q(f"""
        SELECT * FROM {tabla}
        ORDER BY {columna_orden}
        LIMIT {por_pagina} OFFSET {offset}
    """)

for p in range(1, 4):
    print(f"\nPagina {p}:")
    display(paginar("productos", "nombre", p))
```
</details>

---
### Esercizio 3 — Aggregazione

**Consegna:** Con una sola query, calcola:
- Numero totale di ordini.
- Numero di ordini consegnati.
- Numero di ordini in attesa.
- Data dell'ordine più vecchio.
- Data dell'ordine più recente.

Suggerimento: `SUM(estado = 'entregado')` funziona in SQLite (restituisce 1 o 0 per riga).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
display(q("""
    SELECT
        COUNT(*)                           AS total_pedidos,
        SUM(estado = 'entregado')          AS entregados,
        SUM(estado = 'pendiente')          AS pendientes,
        SUM(estado = 'enviado')            AS enviados,
        MIN(fecha)                         AS primer_pedido,
        MAX(fecha)                         AS ultimo_pedido
    FROM pedidos
"""))
```
</details>

---
### Esercizio 4 — DISTINCT e combinazioni

**Consegna:**

1. Quante città distinte ci sono nella tabella clienti?
2. Elenca le combinazioni uniche di (città, prima lettera del nome). Suggerimento: `SUBSTR(nombre, 1, 1)` restituisce il primo carattere.
3. Quanti clienti hanno effettuato ordini? (suggerimento: `COUNT(DISTINCT cliente_id)` sulla tabella pedidos).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
# 1. Numero di città distinte
display(q("SELECT COUNT(DISTINCT ciudad) AS ciudades_distintas FROM clientes"))

# 2. Combinazioni uniche città + iniziale
display(q("""
    SELECT DISTINCT
        ciudad,
        SUBSTR(nombre, 1, 1) AS inicial
    FROM clientes
    ORDER BY ciudad, inicial
"""))

# 3. Clienti che hanno ordinato
display(q("SELECT COUNT(DISTINCT cliente_id) AS clientes_con_pedido FROM pedidos"))
```
</details>

---
### Esercizio 5 (sfida) — Funzione `resumen_tienda`

**Consegna:** Scrivi la funzione `resumen_tienda(conn)` che stampi una dashboard completa del negozio. Deve includere:

- Totale di clienti, prodotti e ordini.
- Valore totale dell'inventario (`precio × stock`).
- Prezzo medio, minimo e massimo dei prodotti.
- Prodotto più caro e prodotto più economico.
- Città con più clienti.
- Percentuale di ordini per stato.

Ogni sezione deve essere separata ed etichettata chiaramente.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

def resumen_tienda(conn):
    def q(sql):
        return pd.read_sql_query(sql, conn)

    print("=" * 50)
    print("    DASHBOARD — NEGOZIO ONLINE")
    print("=" * 50)

    # Totali
    totales = q("""
        SELECT
            (SELECT COUNT(*) FROM clientes)  AS clientes,
            (SELECT COUNT(*) FROM productos) AS productos,
            (SELECT COUNT(*) FROM pedidos)   AS pedidos
    """)
    print("\n--- Totali ---")
    display(totales)

    # Inventario
    inv = q("""
        SELECT
            ROUND(SUM(precio * stock), 2)  AS valor_inventario,
            ROUND(AVG(precio), 2)          AS precio_medio,
            MIN(precio)                    AS precio_min,
            MAX(precio)                    AS precio_max
        FROM productos
    """)
    print("\n--- Inventario ---")
    display(inv)

    # Prodotto più caro / più economico
    extremos = q("""
        SELECT nombre, precio, 'più caro' AS tipo
        FROM productos ORDER BY precio DESC LIMIT 1
        UNION ALL
        SELECT nombre, precio, 'più economico'
        FROM productos ORDER BY precio ASC LIMIT 1
    """)
    print("\n--- Estremi di prezzo ---")
    display(extremos)

    # Città con più clienti
    ciudad = q("""
        SELECT ciudad, COUNT(*) AS clientes
        FROM clientes
        GROUP BY ciudad
        ORDER BY clientes DESC
        LIMIT 1
    """)
    print("\n--- Città con più clienti ---")
    display(ciudad)

    # Ordini per stato
    total = q("SELECT COUNT(*) AS n FROM pedidos").iloc[0]['n']
    estados = q("""
        SELECT estado,
               COUNT(*) AS pedidos
        FROM pedidos
        GROUP BY estado
        ORDER BY pedidos DESC
    """)
    estados['porcentaje'] = (estados['pedidos'] / total * 100).round(1).astype(str) + '%'
    print("\n--- Stati degli ordini ---")
    display(estados)

conn = sqlite3.connect(Path.home() / "tienda.db")
resumen_tienda(conn)
```
</details>

---
## Riepilogo della lezione

```sql
-- Struttura completa di una SELECT
SELECT  [DISTINCT]
        col1 [AS alias],
        ROUND(col2 * 1.21, 2) AS col_calculada,
        COUNT(*) AS total,
        AVG(precio) AS media
FROM    tabla
WHERE   precio BETWEEN 50 AND 500
  AND   nombre LIKE '%Pro%'
  AND   ciudad IN ('Madrid', 'Barcelona')
  AND   descripcion IS NOT NULL
ORDER BY precio DESC, nombre ASC
LIMIT 10 OFFSET 20;   -- pagina 3 di 10 in 10

-- Funzioni di aggregazione
COUNT(*) · COUNT(col) · SUM(col) · AVG(col) · MIN(col) · MAX(col)
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 3** combiniamo le tabelle con `JOIN`, raggruppiamo i risultati con `GROUP BY` e filtriamo i gruppi con `HAVING`.

In [ ]:
conn.close()
print("Connessione chiusa")